# Deep Learning Practice: PyTorch, CNNs, RNNs, Transfer Learning, Transformers & LLMs



## 0. Setup and imports

In [ ]:
# If needed, uncomment and run:
# %pip install torch torchvision scikit-learn matplotlib transformers

import os
import math
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)

## 1. Tensors and autograd
Tensors are multidimensional arrays. Autograd tracks operations and calculates gradients for model training.

In [ ]:
# Tensor creation and basic operations
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones((2, 2))
print("a =\n", a)
print("a + b =\n", a + b)
print("Shape:", a.shape, "| dtype:", a.dtype)

# Autograd example: y = (x^2 + 3x).sum()
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x**2 + 3*x).sum()
y.backward()
print("x:", x.detach())
print("dy/dx:", x.grad)  # derivative is 2x + 3

## 2. A basic model and training loop
We will fit a small neural network to synthetic data. The standard loop is: forward pass → loss → clear gradients → backpropagation → optimizer step.

In [ ]:
# Create a simple two-class synthetic dataset
n_samples = 1200
X = torch.randn(n_samples, 2)
y = ((X[:, 0] * X[:, 1] + 0.35 * X[:, 0]) > 0).long()

perm = torch.randperm(n_samples)
train_idx, test_idx = perm[:960], perm[960:]
train_ds = TensorDataset(X[train_idx], y[train_idx])
test_ds = TensorDataset(X[test_idx], y[test_idx])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128)

class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

model = TinyClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(15):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"Epoch {epoch+1:02d} | train loss: {total_loss/len(train_ds):.4f}")

model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        pred = logits.argmax(dim=1).cpu()
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f"Test accuracy: {correct/total:.3f}")

## 3. CNN for image classification
A convolution layer learns local image patterns; pooling reduces spatial size. This example uses Fashion-MNIST (clothing images).

In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()
try:
    train_img = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
    test_img = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
    train_img_loader = DataLoader(train_img, batch_size=128, shuffle=True)
    test_img_loader = DataLoader(test_img, batch_size=256, shuffle=False)
    fashion_available = True
except Exception as e:
    print("Dataset download failed. Check internet or use Google Colab.", repr(e))
    fashion_available = False

class CNN(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

if fashion_available:
    cnn = CNN().to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(cnn.parameters(), lr=1e-3)
    # A short demonstration run; increase epochs for better performance.
    for epoch in range(3):
        cnn.train()
        running = 0
        for images, labels in train_img_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = loss_fn(cnn(images), labels)
            loss.backward()
            opt.step()
            running += loss.item() * images.size(0)
        print(f"Epoch {epoch+1}: loss={running/len(train_img):.4f}")
else:
    print("CNN class is defined; training skipped because the dataset is unavailable.")

In [ ]:
if fashion_available:
    class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                   "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
    cnn.eval()
    images, labels = next(iter(test_img_loader))
    with torch.no_grad():
        predictions = cnn(images[:8].to(device)).argmax(1).cpu()
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].squeeze(), cmap="gray")
        ax.set_title(f"True: {class_names[labels[i]]}\nPred: {class_names[predictions[i]]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Sequence modeling: RNN, LSTM and GRU
These recurrent layers process a sequence one time step at a time while carrying information forward. LSTM and GRU use gates to better preserve useful information over longer sequences.

In [ ]:
# Compare the basic recurrent layer types on a toy sequence classification task.
# Each example is a sequence of 20 values; label 1 means its mean is positive.
N, T, F = 1500, 20, 1
seq_X = torch.randn(N, T, F)
seq_y = (seq_X.mean(dim=1).squeeze(-1) > 0).long()
idx = torch.randperm(N)
seq_train = TensorDataset(seq_X[idx[:1200]], seq_y[idx[:1200]])
seq_test = TensorDataset(seq_X[idx[1200:]], seq_y[idx[1200:]])
seq_train_loader = DataLoader(seq_train, batch_size=64, shuffle=True)
seq_test_loader = DataLoader(seq_test, batch_size=128)

class SequenceClassifier(nn.Module):
    def __init__(self, kind="LSTM", hidden=32):
        super().__init__()
        layers = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
        self.rnn = layers[kind](input_size=1, hidden_size=hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 2)
    def forward(self, x):
        out, state = self.rnn(x)
        last = out[:, -1, :]
        return self.fc(last)

def train_sequence_model(kind, epochs=5):
    net = SequenceClassifier(kind).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=0.003)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        net.train()
        for xb, yb in seq_train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(net(xb), yb)
            loss.backward()
            opt.step()
    net.eval()
    right = count = 0
    with torch.no_grad():
        for xb, yb in seq_test_loader:
            pred = net(xb.to(device)).argmax(1).cpu()
            right += (pred == yb).sum().item()
            count += len(yb)
    return right / count

for kind in ["RNN", "LSTM", "GRU"]:
    print(f"{kind} test accuracy: {train_sequence_model(kind):.3f}")

## 5. Regularization and early stopping
- **Dropout:** randomly disables some activations during training.
- **Batch normalization:** normalizes intermediate activations and often stabilizes training.
- **Early stopping:** stops when validation loss stops improving, reducing overfitting.

The CNN above already includes dropout and batch normalization. The reusable early-stopping example follows.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.bad_epochs = 0
        self.best_state = None
    def step(self, val_loss, model):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.bad_epochs = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

# Example usage in a training loop:
# stopper = EarlyStopping(patience=3)
# for epoch in range(max_epochs):
#     train_one_epoch(...)
#     val_loss = validate(...)
#     if stopper.step(val_loss, model):
#         print("Early stopping")
#         model.load_state_dict(stopper.best_state)
#         break
print("EarlyStopping helper is ready to use.")

## 6. Transfer learning with a pretrained CNN
Transfer learning reuses features learned from a large dataset. This example loads pretrained ResNet-18 and replaces its final layer for a new number of classes. It shows the setup; training requires your own labeled dataset and internet access to download weights.

In [ ]:
from torchvision import models

try:
    weights = models.ResNet18_Weights.DEFAULT
    resnet = models.resnet18(weights=weights)
    num_classes = 2  # change to match your dataset
    in_features = resnet.fc.in_features
    resnet.fc = nn.Linear(in_features, num_classes)

    # Freeze pretrained feature extractor; train only the new classifier first.
    for name, param in resnet.named_parameters():
        param.requires_grad = name.startswith("fc.")

    resnet = resnet.to(device)
    transfer_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-3
    )
    preprocess = weights.transforms()
    print("Pretrained ResNet-18 ready.")
    print("Expected input transform:", preprocess)
    print("Trainable parameters:", sum(p.numel() for p in resnet.parameters() if p.requires_grad))
except Exception as e:
    print("Could not load pretrained weights. Check internet / torchvision setup.", repr(e))

### Transfer learning training template

Prepare a dataset in `ImageFolder` format:

```
dataset/
  train/
    class_a/
    class_b/
  val/
    class_a/
    class_b/
```

Then use the pretrained model's transform and train the final layer.

In [ ]:
# Uncomment after preparing the folders above.
# from torchvision.datasets import ImageFolder
# from torch.utils.data import DataLoader
#
# train_data = ImageFolder("dataset/train", transform=preprocess)
# val_data = ImageFolder("dataset/val", transform=preprocess)
# train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_data, batch_size=32)
#
# criterion = nn.CrossEntropyLoss()
# for epoch in range(5):
#     resnet.train()
#     for xb, yb in train_loader:
#         xb, yb = xb.to(device), yb.to(device)
#         transfer_optimizer.zero_grad()
#         loss = criterion(resnet(xb), yb)
#         loss.backward()
#         transfer_optimizer.step()
#     print("Finished epoch", epoch + 1)


## 7. Hugging Face Transformers: pretrained NLP
We will use a pretrained sentiment-analysis pipeline. The first run may download model files.

In [ ]:
# If needed, uncomment:
# %pip install transformers

try:
    from transformers import pipeline
    sentiment = pipeline("sentiment-analysis")
    samples = [
        "This deep learning notebook is helpful and easy to follow.",
        "The installation failed and the results were disappointing."
    ]
    for text, result in zip(samples, sentiment(samples)):
        print(text, "->", result)
except Exception as e:
    print("Transformers demo could not run. Install transformers and check internet.", repr(e))

### Optional: text generation with a pretrained transformer
Text generation can require more memory and may download a larger model. This compact model is for experimentation, not production use.

In [ ]:
# Uncomment to try text generation:
# from transformers import pipeline
# generator = pipeline("text-generation", model="distilgpt2")
# result = generator("Artificial intelligence can help students by", max_new_tokens=50,
#                    do_sample=True, temperature=0.8, pad_token_id=50256)
# print(result[0]["generated_text"])

## 8. Using LLM APIs or local open models
Choose one route. Keep API keys private: never paste them into a public notebook or commit them to GitHub.

### Option A — OpenAI API (requires an API key and billing access)

In [ ]:
# Install if needed:
# %pip install openai
#
# Set OPENAI_API_KEY in your environment (recommended) before running:
# import os
# from openai import OpenAI
# client = OpenAI()  # reads OPENAI_API_KEY from environment
#
# response = client.responses.create(
#     model="gpt-4.1-mini",
#     input="Summarize in three bullet points: Neural networks learn patterns from data."
# )
# print(response.output_text)

### Option B — Anthropic API (requires an API key)

In [ ]:
# Install if needed:
# %pip install anthropic
#
# import os
# from anthropic import Anthropic
# client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
# message = client.messages.create(
#     model="claude-3-5-haiku-latest",
#     max_tokens=150,
#     messages=[{"role": "user", "content": "Explain CNNs in simple words."}]
# )
# print(message.content[0].text)

### Option C — Ollama local model
Install Ollama separately, download a model such as `llama3.2`, and start the local Ollama service before running this cell.

In [ ]:
# %pip install ollama
#
# import ollama
# response = ollama.chat(
#     model="llama3.2",
#     messages=[{"role": "user", "content": "Explain transfer learning in 4 sentences."}]
# )
# print(response["message"]["content"])